# Labelling the data with next best skill and next role

## Attaching next skill to each data point

In [ ]:
import json
from pathlib import Path
from collections import Counter
from clean_designations import get_matching_occupations

# Paths
data_dir = Path("data")
input_file = "./data/processed/essential_resumes.json"
output_file = "./data/processed/designation_aggregated_skills.json"

# Get occupations
occupations = get_matching_occupations(
    data_dir / "acs_it_occupations.csv",
    data_dir / "anzsco_occupations.csv"
)

# If occupations is a DataFrame, extract the designation column
if hasattr(occupations, "columns"):
    designation_list = occupations["name"].dropna().unique().tolist()
else:
    designation_list = list(occupations)

print(len(designation_list))

# Load resumes
with open(input_file, "r", encoding="utf-8") as f:
    resumes = json.load(f)

designation_data = {}

for designation in designation_list:

    # If designation_list contains dicts
    if isinstance(designation, dict):
        designation_name = designation["name"]
    else:
        designation_name = designation

    aggregated = {
        "skills": Counter(),
        "it_skills": Counter(),
        "it_skill_categories": Counter(),
        "soft_skills": Counter(),
        "languages": Counter(),
    }

    # Filter resumes that contain this designation
    for resume in resumes:
        resume_designations = resume.get("designations", [])

        if resume_designations and designation_name in resume_designations:
            for field in aggregated.keys():
                values = resume.get(field, [])
                if isinstance(values, list):
                    aggregated[field].update(values)

    # Convert Counters to normal dicts for JSON serialization
    if any(aggregated.values()):
        designation_data[designation_name] = {
            field: dict(counter)
            for field, counter in aggregated.items()
        }

print(len(designation_data))

# Save results
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(designation_data, f, indent=4, ensure_ascii=False)

print(f"Saved weighted aggregated skills for {len(designation_data)} designations to '{output_file}'")


120
14
Saved weighted aggregated skills for 14 designations to './data/processed/designation_aggregated_skills.json'


/Users/hammadhassan/Documents/semester4/career-mentor-ai/resume_processor/scripts/extract_occupations.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  anzsco_df = anzsco_df[anzsco_df["Occupation Code"].str.isdigit().fillna(False)]


## Generating training data

Resumes are duplicated against each designation and output labels of next_skill and next_soft_skill are added as output labels.

In [22]:
import json
from pathlib import Path

# Paths
resume_file = "./data/processed/essential_resumes.json"
designation_file = "./data/processed/designation_aggregated_skills.json"
output_file = "./data/processed/training_data.json"

# Load resumes
with open(resume_file, "r", encoding="utf-8") as f:
    resumes = json.load(f)

# Load designation aggregated skills
with open(designation_file, "r", encoding="utf-8") as f:
    designation_data = json.load(f)

training_data = []

# For each resume
for resume in resumes:
    current_it_skills = set(resume.get("it_skill_categories", []))
    current_soft_skills = set(resume.get("soft_skills", []))

    # For each possible designation
    for designation_name, skills_data in designation_data.items():

        # Determine missing IT skills
        designation_it_skills = skills_data.get("it_skill_categories", {})
        next_skill = {
            skill: count
            for skill, count in designation_it_skills.items()
            if skill not in current_it_skills
        }

        # Determine missing soft skills
        designation_soft_skills = skills_data.get("soft_skills", {})
        next_soft_skill = {
            skill: count
            for skill, count in designation_soft_skills.items()
            if skill not in current_soft_skills
        }

        # Prepare the training example
        example = {
            "it_skill_categories": list(current_it_skills),
            "soft_skills": list(current_soft_skills),
            "desired_designation": designation_name,
            "next_skill": next_skill,           # keep counts
            "next_soft_skill": next_soft_skill  # keep counts
        }

        training_data.append(example)

# Save the training data
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(training_data, f, indent=4, ensure_ascii=False)

print(f"Generated {len(training_data)} training examples in '{output_file}'")


Generated 1400 training examples in './data/processed/training_data.json'
